In [ ]:
import os
from dotenv import load_dotenv
from roboflow import Roboflow

nombre_carpeta = "deteccion_defectos-1"

# os.path.isdir verifica la existencia y que sea una carpeta al mismo tiempo
if os.path.isdir(nombre_carpeta):
    print(f"¡La carpeta '{nombre_carpeta}' existe!")
else:
    print(f"La carpeta '{nombre_carpeta}' no existe, se descargará de roboflow.")
    # Carga las variables del archivo .env
    load_dotenv()
    
    # Obtiene la API Key de las variables de entorno
    ROBOFLOW_KEY = os.getenv("API_ROBOFLOW")
    ROBOFLOW_WORKSPACE = os.getenv("API_ROBOFLOW_WORKSPACE")
    
    if not ROBOFLOW_KEY:
        raise ValueError("No se encontró la variable de entorno 'API_ROBOFLOW'. Asegúrate de definirla en tu archivo .env")
    rf = Roboflow(api_key=f"{ROBOFLOW_KEY}")
    project = rf.workspace("clothesdataset-yimbv").project("deteccion_defectos")
    version = project.version(1)
    dataset = version.download("yolov11")
    print(f"Dataset descargado de roboflow")

In [1]:
import yaml
import os

nombre_carpeta = 'deteccion_defectos-1'
# Define las rutas absolutas o relativas desde donde ejecutarás el entrenamiento
# Es mejor usar rutas completas para evitar errores de "File Not Found"
dataset_path = os.path.abspath(f"{nombre_carpeta}")

data_config = {
    'path': dataset_path,      # Directorio raíz del dataset
    'train': 'train/images',   # Ruta relativa a 'path' para entrenamiento
    'val': 'valid/images',     # Ruta relativa a 'path' para validación
    'test': 'test/images',     # Ruta relativa a 'path' para pruebas (opcional)

    'nc': 2,                   # Número de clases
    'names': ['Corrido', 'Hueco'] # Asegúrate de que este orden sea el mismo de Roboflow
}

# Guardar el archivo
with open('dataset.yaml', 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)

print("Archivo dataset.yaml creado con éxito.")

Archivo dataset.yaml creado con éxito.


In [9]:
import torch
from ultralytics import YOLO
import os
import gc

def clear_gpu():
    torch.cuda.empty_cache()
    gc.collect()
    # Forzamos a que PyTorch gestione mejor la fragmentación en la serie 40/50
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# 1. ENTORNO Y RUTAS
device = 0 if torch.cuda.is_available() else "cpu"
DATASET_YAML = os.path.abspath("dataset.yaml")
BASE_MODEL = "yolo11m.pt"

# 2. DEFINICIÓN DEL PIPELINE DE ALBUMENTATIONS
try:
    import albumentations as A
    
    custom_transform = A.Compose([
        # CLAHE para mejorar el contraste de hilos y huecos en prendas beige/marrones
        A.CLAHE(clip_limit=3.0, tile_grid_size=(8, 8), p=0.3),
        # Resistencia al ruido electrónico de la cámara industrial
        A.GaussNoise(std_limit=(10.0, 30.0), p=0.2),
        # Desenfoque leve por vibraciones físicas de la estructura de captura
        A.Blur(blur_limit=3, p=0.1),
    ], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))

except ImportError:
    print("❌ Error: Instala albumentations usando 'pip install albumentations'")
    exit(1)


# 3. CALLBACK DE INYECCIÓN LIMPIA EN MEMORIA
def inject_custom_albumentations(trainer):
    """
    Este callback modifica directamente el loader una vez armado,
    reemplazando el pipeline interno por nuestro set industrial.
    """
    print("\n⚡ [MLOps] Sincronizando datasets con Albumentations Industrial...")
    
    if hasattr(trainer, 'train_loader') and trainer.train_loader is not None:
        dataset = trainer.train_loader.dataset
        # Buscamos el wrapper de Albumentations dentro de los transforms nativos
        for transform in dataset.transforms.transforms:
            if type(transform).__name__ == 'Albumentations':
                transform.transform = custom_transform
                print("✅ Albumentations personalizado inyectado correctamente en el Loader.")


# 📊 DICCIONARIO DE AUGMENTATION NATIVO REUTILIZABLE
# Se aplica a ambas fases para asegurar una base libre de sobreajuste estructural
INDUSTRIAL_AUGMENTATION = {
    "hsv_v": 0.4,          # Variabilidad de brillo (manejo de sombras en arrugas)
    "hsv_s": 0.3,          # Variabilidad de saturación
    "degrees": 10.0,       # Rotaciones leves para deformación del tejido
    "scale": 0.15,         # Escalamiento dinámico para cajas < 50px
    "perspective": 0.0005, # Perspectiva sutil
    "mosaic": 0.4,         # Control de mosaico para no sobre-comprimir texturas
    "copy_paste": 0.6,     # Forzar duplicación de bboxes críticas de "Hueco"
    "mixup": 0.1,          # Superposición para robustecer el modelo contra grecas complejas
    "erasing": 0.4         # 🎯 CRÍTICO: Tapa 40% del tejido para forzar generalización contextual
}


# 4. EJECUCIÓN DEL ENTRENAMIENTO
def run_industrial_training():
    # --- 🧊 FASE 1: Entrenando solo la Cabeza de Detección (Freeze Backbone) ---
    print("\n--- 🧊 FASE 1: Entrenando solo la Cabeza de Detección (Freeze Backbone) ---")
    model_fase1 = YOLO(BASE_MODEL)
    
    # IMPORTANTE: El callback se registra en esta instancia específica
    model_fase1.add_callback("on_train_start", inject_custom_albumentations)
    
    fase1_results = model_fase1.train(
        data=DATASET_YAML,
        epochs=30,  # Épocas cortas de calentamiento para la cabeza
        imgsz=1280,
        batch=4,           
        amp=True,
        workers=4,
        freeze=12,  # Congela el backbone convolucional base
        optimizer="AdamW",
        lr0=1e-4,
        weight_decay=0.05,
        device=device,
        project="chompas_quality_control",
        name="fase1_frozen_v3",
        **INDUSTRIAL_AUGMENTATION  # Inyectamos los mismos aumentos nativos
    )
    
    checkpoint_fase1_path = os.path.join(fase1_results.save_dir, "weights", "last.pt")
    
    # Liberamos memoria de la fase 1 de forma explícita antes de instanciar la fase 2
    del model_fase1
    clear_gpu()
    
    # --- 🔥 FASE 2: Fine-Tuning de Red Completa (Unfreeze) ---
    print("\n--- 🔥 FASE 2: Fine-Tuning de Red Completa (Unfreeze) ---")
    model_fase2 = YOLO(checkpoint_fase1_path)
    
    # 🔥 RE-REGISTRO CRÍTICO: Al instanciar desde un archivo físico, 
    # debemos volver a enlazar el callback en memoria para esta fase.
    model_fase2.add_callback("on_train_start", inject_custom_albumentations)

    model_fase2.train(
        data=DATASET_YAML,
        epochs=170,
        imgsz=1280,
        batch=4,
        amp=True,
        workers=4,
        freeze=None,  # Descongelamos todo el modelo para ajuste fino profundo
        optimizer="AdamW",
        lr0=5e-5,
        patience=45,
        box=12.0,     # Penalización geométrica alta para delimitar bien el defecto
        cls=4.0,      # Penalización de clasificación masiva contra falsos positivos
        dfl=2.5,
        project="chompas_quality_control",
        name="fase2_unfrozen_v3_augmentation",
        plots=True,
        **INDUSTRIAL_AUGMENTATION  # Mantenemos consistencia en el augmentation
    )

if __name__ == "__main__":
    clear_gpu()
    run_industrial_training()

C:\Users\USER\AppData\Local\Temp\ipykernel_15864\1657028177.py:25: UserWarning: Argument(s) 'std_limit' are not valid for transform GaussNoise
  A.GaussNoise(std_limit=(10.0, 30.0), p=0.2),
C:\Users\USER\Desktop\clothes-failures-detection\.venv\Lib\site-packages\albumentations\core\composition.py:331: UserWarning: Got processor for bboxes, but no transform to process it.
  self._set_keys()



--- 🧊 FASE 1: Entrenando solo la Cabeza de Detección (Freeze Backbone) ---
New https://pypi.org/project/ultralytics/8.4.87 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.241  Python-3.12.12 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5060, 8151MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.6, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\USER\Desktop\clothes-failures-detection\Notebooks\dataset.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=12, half=False, hsv_h=0.015, hsv_s=0.3, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, 

In [14]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch
from ultralytics import YOLO
import gc

def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()

clear_gpu()

ruta_mejor_modelo = r"C:\Users\USER\Desktop\clothes-failures-detection\Notebooks\chompas_quality_control\fase2_unfrozen_v3_augmentation\weights\best.pt"

model = YOLO(ruta_mejor_modelo)

metrics = model.val(conf=0.3)
print(metrics.results_dict['metrics/precision(B)'])
print(metrics.results_dict['metrics/recall(B)'])

Ultralytics 8.3.241  Python-3.12.12 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5060, 8151MiB)
YOLO11m summary (fused): 125 layers, 20,031,574 parameters, 0 gradients, 67.7 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 3817.5929.3 MB/s, size: 601.5 KB)
val: Scanning C:\Users\USER\Desktop\clothes-failures-detection\Notebooks\deteccion_defectos-1\valid\labels.cache... 20 images, 5 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 20/20 39.8Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 1.3s/it 2.7s8.3s
                   all         20        381      0.773      0.627      0.703      0.343
               Corrido         13        295      0.837      0.661      0.781      0.415
                 Hueco         13         86      0.708      0.593      0.626      0.271
Speed: 6.5ms preprocess, 24.5ms inference, 0.0ms loss, 0.4ms postprocess per image
Results saved to C:\Users\USER\Desktop\clothes-failures-detection

In [15]:
# Corre la validación con una confianza muy baja y evalúa el comportamiento
metrics_low_conf = model.val(conf=0.10, iou=0.45)

print("--- METRICAS A BAJA CONFIANZA (Operativo para priorizar Recall) ---")
print(f"Precision General: {metrics_low_conf.results_dict['metrics/precision(B)']:.4f}")
print(f"Recall General: {metrics_low_conf.results_dict['metrics/recall(B)']:.4f}")

Ultralytics 8.3.241  Python-3.12.12 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5060, 8151MiB)
val: Fast image access  (ping: 0.00.0 ms, read: 2932.8570.5 MB/s, size: 550.3 KB)
val: Scanning C:\Users\USER\Desktop\clothes-failures-detection\Notebooks\deteccion_defectos-1\valid\labels.cache... 20 images, 5 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 20/20  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 1.3s/it 2.7s8.2s
                   all         20        381      0.718       0.71      0.735      0.339
               Corrido         13        295      0.778      0.723      0.799      0.403
                 Hueco         13         86      0.659      0.698       0.67      0.274
Speed: 5.9ms preprocess, 23.9ms inference, 0.0ms loss, 0.5ms postprocess per image
Results saved to C:\Users\USER\Desktop\clothes-failures-detection\runs\detect\val24
--- METRICAS A BAJA CONFIANZA (Operativo para priorizar Recall) ---
Precisi

In [4]:
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
import os

# Configurar el modelo de Ultralytics dentro de SAHI
detection_model = AutoDetectionModel.from_pretrained(
    model_type='yolov11',
    model_path=ruta_mejor_modelo,
    confidence_threshold=0.15, # Umbral optimizado para Recall
    device="cuda:0"            # Tu RTX 5060
)

# Prueba con una imagen real de validación que sepas que tiene "Huecos"
# Cambia esta ruta por una imagen real de tu set de validación
ruta_imagen_prueba = r"C:\Users\USER\Desktop\clothes-failures-detection\Notebooks\deteccion_defectos-1\valid\images\tmp0j228_tu_jpg.rf.1fd7fba19cbad9be342075238fe4f657.jpg"

result = get_sliced_prediction(
    ruta_imagen_prueba,
    detection_model,
    slice_height=640,     # Alto del parche (mantiene resolución nativa de los hilos)
    slice_width=640,      # Ancho del parche
    overlap_height_ratio=0.2, # 20% de solapamiento para no cortar defectos en los bordes
    overlap_width_ratio=0.2
)

# Guarda el resultado visual en tu disco para auditarlo
result.export_visuals(export_dir=".", file_name="resultado_sahi")
print("⚡ Inferencia por parches completada. Revisa 'resultado_sahi.png' para verificar si capturó el hueco.")

Performing prediction on 32 slices.
⚡ Inferencia por parches completada. Revisa 'resultado_sahi.png' para verificar si capturó el hueco.


In [21]:
import os
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from sahi.utils.coco import Coco
from sahi.utils.file import load_json
import numpy as np

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch
from ultralytics import YOLO
import gc

def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()

clear_gpu()

# 1. CONFIGURACIÓN DE RUTAS (Ajusta a tu entorno local)
ruta_mejor_modelo = r"C:\Users\USER\Desktop\clothes-failures-detection\Notebooks\chompas_quality_control\fase2_unfrozen_v3_augmentation\weights\best.pt"
dir_valid_images = r"C:\Users\USER\Desktop\clothes-failures-detection\Notebooks\deteccion_defectos-1\valid\images"
dir_valid_labels = r"C:\Users\USER\Desktop\clothes-failures-detection\Notebooks\deteccion_defectos-1\valid\labels"

# Mapeo exacto de tus clases
class_map = {0: "Corrido", 1: "Hueco"}

# 2. CARGAR MODELO EN SAHI
detection_model = AutoDetectionModel.from_pretrained(
    model_type='yolov11',
    model_path=ruta_mejor_modelo,
    confidence_threshold=0.30, # Umbral sugerido para balancear falsos positivos
    device="cuda:0"
)

# 3. BUCLE DE EVALUACIÓN INDUSTRIAL
print("🚀 Iniciando evaluación global con SAHI en resolución nativa...")

true_positives = {"Corrido": 0, "Hueco": 0}
false_positives = {"Corrido": 0, "Hueco": 0}
false_negatives = {"Corrido": 0, "Hueco": 0}

lista_imagenes = [f for f in os.listdir(dir_valid_images) if f.endswith(('.jpg', '.jpeg', '.png'))]

for img_name in lista_imagenes:
    img_path = os.path.join(dir_valid_images, img_name)
    lbl_path = os.path.join(dir_valid_labels, os.path.splitext(img_name)[0] + ".txt")
    
    # A. Inferencia por parches con SAHI
    prediction = get_sliced_prediction(
        img_path,
        detection_model,
        slice_height=1920,
        slice_width=1920,
        overlap_height_ratio=0.35,
        overlap_width_ratio=0.35,
        verbose=0
    )
    
    # Extraer bboxes predichas por SAHI
    preds = []
    for p in prediction.object_prediction_list:
        # SAHI entrega [xmin, ymin, xmax, ymax]
        preds.append({
            "box": p.bbox.to_voc_bbox(), 
            "class": class_map[p.category.id]
        })
        
    # B. Leer etiquetas reales (Ground Truth) y desnormalizar coordenadas YOLO
    gts = []
    if os.path.exists(lbl_path):
        from PIL import Image
        with Image.open(img_path) as img:
            w_img, h_img = img.size
            
        with open(lbl_path, "r") as f:
            for line in f.readlines():
                parts = line.strip().split()
                cls_id = int(parts[0])
                cx, cy, w, h = map(float, parts[1:])
                # Convertir de YOLO normalizado a VOC absoluto [xmin, ymin, xmax, ymax]
                xmin = int((cx - w/2) * w_img)
                ymin = int((cy - h/2) * h_img)
                xmax = int((cx + w/2) * w_img)
                ymax = int((cy + h/2) * h_img)
                gts.append({"box": [xmin, ymin, xmax, ymax], "class": class_map[cls_id]})

    # C. Matchear Predicciones vs Ground Truth usando IoU simple para Métricas
    def calcular_iou(boxA, boxB):
        xA = max(boxA[0], boxB[0])
        yA = max(boxA[1], boxB[1])
        xB = min(boxA[2], boxB[2])
        yB = min(boxA[3], boxB[3])
        interArea = max(0, xB - xA) * max(0, yB - yA)
        boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
        boxBAArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
        iou = interArea / float(boxAArea + boxBAArea - interArea) if (boxAArea + boxBAArea - interArea) > 0 else 0
        return iou

    matched_gts = set()
    for p in preds:
        best_iou = 0
        best_gt_idx = -1
        for idx, gt in enumerate(gts):
            if p["class"] == gt["class"] and idx not in matched_gts:
                iou = calcular_iou(p["box"], gt["box"])
                if iou > best_iou:
                    best_iou = iou
                    best_gt_idx = idx
                    
        if best_iou >= 0.30: # Umbral IoU estándar de aceptación
            true_positives[p["class"]] += 1
            matched_gts.add(best_gt_idx)
        else:
            false_positives[p["class"]] += 1
            
    for idx, gt in enumerate(gts):
        if idx not in matched_gts:
            false_negatives[gt["class"]] += 1

# 4. CALCULAR Y MOSTRAR REPORTES EN CONSOLA
print("\n" + "="*45)
print("📊 REPORTE DE MÉTRICAS INDUSTRIALES (SAHI 4K)")
print("="*45)

for clase in ["Corrido", "Hueco"]:
    tp = true_positives[clase]
    fp = false_positives[clase]
    fn = false_negatives[clase]
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    print(f"\n🟢 CLASE: {clase.upper()}")
    print(f"  ▪️ True Positives (Detectados): {tp}")
    print(f"  ▪️ False Positives (Falsas Alarmas): {fp}")
    print(f"  ▪️ False Negatives (Omitidos): {fn}")
    print(f"  ➡️ Precisión: {precision:.4f}")
    print(f"  ➡️ Recall: {recall:.4f}  <-- Objetivo >0.90")
    print(f"  ➡️ F1-Score: {f1:.4f}")

print("\n" + "="*45)

🚀 Iniciando evaluación global con SAHI en resolución nativa...

📊 REPORTE DE MÉTRICAS INDUSTRIALES (SAHI 4K)

🟢 CLASE: CORRIDO
  ▪️ True Positives (Detectados): 239
  ▪️ False Positives (Falsas Alarmas): 73
  ▪️ False Negatives (Omitidos): 56
  ➡️ Precisión: 0.7660
  ➡️ Recall: 0.8102  <-- Objetivo >0.90
  ➡️ F1-Score: 0.7875

🟢 CLASE: HUECO
  ▪️ True Positives (Detectados): 66
  ▪️ False Positives (Falsas Alarmas): 31
  ▪️ False Negatives (Omitidos): 20
  ➡️ Precisión: 0.6804
  ➡️ Recall: 0.7674  <-- Objetivo >0.90
  ➡️ F1-Score: 0.7213



In [19]:
import os
import cv2
from PIL import Image
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

# =====================================================================
# 1. CONFIGURACIÓN DE RUTAS Y PARÁMETROS DE NEGOCIO
# =====================================================================
ruta_mejor_modelo = r"C:\Users\USER\Desktop\clothes-failures-detection\Notebooks\chompas_quality_control\fase2_unfrozen_v3_augmentation\weights\best.pt"
dir_valid_images = r"C:\Users\USER\Desktop\clothes-failures-detection\Notebooks\deteccion_defectos-1\valid\images"
dir_valid_labels = r"C:\Users\USER\Desktop\clothes-failures-detection\Notebooks\deteccion_defectos-1\valid\labels"
output_audit_dir = r"C:\Users\USER\Desktop\clothes-failures-detection\Notebooks\auditoria_errores"

os.makedirs(output_audit_dir, exist_ok=True)

# Mapeo de clases de Roboflow
class_map = {0: "Corrido", 1: "Hueco"}

# 🎯 UMBRALES ESPECÍFICOS POR CLASE (Para recuperar Precisión)
CONF_THRESHOLDS = {
    "Corrido": 0.25,  # Más estricto porque es una clase con más datos
    "Hueco": 0.20     # Filtro intermedio para limpiar ruido sin perder el Recall ganado
}

# 📏 FILTRO FÍSICO (Dimensiones mínimas en píxeles para ignorar ruido del tejido)
MIN_DIMS = {
    "Corrido": 10,    # Ancho o alto mínimo para hilos corridos
    "Hueco": 10       # Un hueco real en 4K debe medir al menos 25x25 píxeles
}

# =====================================================================
# 2. CARGAR MODELO EN SAHI
# =====================================================================
# Se inicializa con el umbral más bajo para permitir que el pipeline evalúe los candidatos
detection_model = AutoDetectionModel.from_pretrained(
    model_type='yolov11',
    model_path=ruta_mejor_modelo,
    confidence_threshold=0.20, 
    device="cuda:0"
)

# Contadores de Matriz de Confusión
true_positives = {"Corrido": 0, "Hueco": 0}
false_positives = {"Corrido": 0, "Hueco": 0}
false_negatives = {"Corrido": 0, "Hueco": 0}

lista_imagenes = [f for f in os.listdir(dir_valid_images) if f.endswith(('.jpg', '.jpeg', '.png'))]

print(f"🚀 Iniciando evaluación industrial sobre {len(lista_imagenes)} imágenes...")

# =====================================================================
# 3. BUCLE PRINCIPAL DE EVALUACIÓN
# =====================================================================
for img_name in lista_imagenes:
    img_path = os.path.join(dir_valid_images, img_name)
    lbl_path = os.path.join(dir_valid_labels, os.path.splitext(img_name)[0] + ".txt")
    
    # A. Inferencia por Parches Sincronizados (Escala 1920px para emular entrenamiento)
    prediction = get_sliced_prediction(
        img_path,
        detection_model,
        slice_height=1920,
        slice_width=1920,
        overlap_height_ratio=0.35,  # Evita cortes perimetrales en los bordes del parche
        overlap_width_ratio=0.35,
        postprocess_type="NMS",     # Combina de forma estricta bboxes redundantes
        postprocess_match_threshold=0.5,
        verbose=0
    )
    
    # B. Procesamiento y Filtrado Post-Inferencia
    preds = []
    for p in prediction.object_prediction_list:
        clase_detectada = class_map[p.category.id]
        confianza = p.score.value
        box_voc = p.bbox.to_voc_bbox()  # [xmin, ymin, xmax, ymax]
        
        # Calcular dimensiones físicas de la predicción
        ancho_box = box_voc[2] - box_voc[0]
        alto_box = box_voc[3] - box_voc[1]
        
        # 🛡️ Aplicar Filtro de Confianza por Clase y Filtro de Tamaño Mínimo
        if confianza >= CONF_THRESHOLDS[clase_detectada]:
            if ancho_box >= MIN_DIMS[clase_detectada] and alto_box >= MIN_DIMS[clase_detectada]:
                preds.append({
                    "box": box_voc, 
                    "class": clase_detectada
                })
        
    # C. Leer y desnormalizar las etiquetas reales (Ground Truth)
    gts = []
    if os.path.exists(lbl_path):
        with Image.open(img_path) as img:
            w_img, h_img = img.size
            
        with open(lbl_path, "r") as f:
            for line in f.readlines():
                parts = line.strip().split()
                if not parts: continue
                cls_id = int(parts[0])
                cx, cy, w, h = map(float, parts[1:])
                
                # Conversión de formato YOLO normalizado a VOC absoluto
                xmin = int((cx - w/2) * w_img)
                ymin = int((cy - h/2) * h_img)
                xmax = int((cx + w/2) * w_img)
                ymax = int((cy + h/2) * h_img)
                gts.append({"box": [xmin, ymin, xmax, ymax], "class": class_map[cls_id]})

    # D. Función matemática para emparejamiento por IoU
    def calcular_iou(boxA, boxB):
        xA = max(boxA[0], boxB[0])
        yA = max(boxA[1], boxB[1])
        xB = min(boxA[2], boxB[2])
        yB = min(boxA[3], boxB[3])
        interArea = max(0, xB - xA) * max(0, yB - yA)
        boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
        boxBAArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
        return interArea / float(boxAArea + boxBAArea - interArea) if (boxAArea + boxBAArea - interArea) > 0 else 0

    # E. Macheo de Cajas para Métricas (Preds vs GT)
    matched_gts = set()
    for p in preds:
        best_iou = 0
        best_gt_idx = -1
        for idx, gt in enumerate(gts):
            if p["class"] == gt["class"] and idx not in matched_gts:
                iou = calcular_iou(p["box"], gt["box"])
                if iou > best_iou:
                    best_iou = iou
                    best_gt_idx = idx
                    
        if best_iou >= 0.30:  # Tolerancia espacial industrial adaptativa
            true_positives[p["class"]] += 1
            matched_gts.add(best_gt_idx)
        else:
            false_positives[p["class"]] += 1
            
    for idx, gt in enumerate(gts):
        if idx not in matched_gts:
            false_negatives[gt["class"]] += 1

    # F. Exportación Automática de Falsos Negativos (Auditoría Visual)
    hubo_omision = len(gts) > len(matched_gts)
    if hubo_omision:
        img_visual = cv2.imread(img_path)
        for idx, gt in enumerate(gts):
            if idx not in matched_gts:
                box = gt["box"]
                # Pintar defecto OMITIDO en ROJO en la imagen original
                cv2.rectangle(img_visual, (box[0], box[1]), (box[2], box[3]), (0, 0, 255), 3)
                cv2.putText(
                    img_visual, f"OMITIDO: {gt['class']}", 
                    (box[0], box[1] - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2
                )
        audit_save_path = os.path.join(output_audit_dir, f"error_{img_name}")
        cv2.imwrite(audit_save_path, img_visual)

# =====================================================================
# 4. REPORTE FINAL DE MÉTRICAS CONSOLIDADAS
# =====================================================================
print("\n" + "="*50)
print("📊 REPORTEN DE MÉTRICAS FILTRADAS (SAHI 4K + POST-PROCESS)")
print("="*50)

for clase in ["Corrido", "Hueco"]:
    tp = true_positives[clase]
    fp = false_positives[clase]
    fn = false_negatives[clase]
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    print(f"\n🟢 CLASE: {clase.upper()}")
    print(f"  ▪️ True Positives (Detectados): {tp}")
    print(f"  ▪️ False Positives (Falsas Alarmas Limpias): {fp}")
    print(f"  ▪️ False Negatives (Omitidos Reales): {fn}")
    print(f"  ➡️ Precisión Recuperada: {precision:.4f}")
    print(f"  ➡️ Recall Sostenido: {recall:.4f}")
    print(f"  ➡️ F1-Score: {f1:.4f}")

print("\n" + "="*50)
print(f"📁 Las imágenes con los {false_negatives['Corrido'] + false_negatives['Hueco']} defectos omitidos fueron guardadas en: {output_audit_dir}")

🚀 Iniciando evaluación industrial sobre 20 imágenes...

📊 REPORTEN DE MÉTRICAS FILTRADAS (SAHI 4K + POST-PROCESS)

🟢 CLASE: CORRIDO
  ▪️ True Positives (Detectados): 250
  ▪️ False Positives (Falsas Alarmas Limpias): 89
  ▪️ False Negatives (Omitidos Reales): 45
  ➡️ Precisión Recuperada: 0.7375
  ➡️ Recall Sostenido: 0.8475
  ➡️ F1-Score: 0.7886

🟢 CLASE: HUECO
  ▪️ True Positives (Detectados): 72
  ▪️ False Positives (Falsas Alarmas Limpias): 48
  ▪️ False Negatives (Omitidos Reales): 14
  ➡️ Precisión Recuperada: 0.6000
  ➡️ Recall Sostenido: 0.8372
  ➡️ F1-Score: 0.6990

📁 Las imágenes con los 59 defectos omitidos fueron guardadas en: C:\Users\USER\Desktop\clothes-failures-detection\Notebooks\auditoria_errores
